# RAG krok po kroku

Ten sam pipeline co `main.py`, rozpisany na etapy: PDF → chunki → embeddingi → Chroma → wyszukiwanie → prompt → odpowiedź.

Wymaga działającej Ollamy z modelami `bge-m3` i `qwen3:4b`. Notebook przebudowuje indeks (jak `python main.py reset` + `add`).

In [1]:
import os
import sys
from pathlib import Path

# notebook leży w notebooks/, a kod importujemy z katalogu projektu
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src import config

## 1. Tekst z PDF

`load_documents()` wyciąga tekst przez `pypdf`. Widać strukturę, na której opiera się chunker: tytuł, linia `Status: …` i krótkie nagłówki sekcji. PDF łamie zdania na kilka linii.

In [2]:
from src.loader import load_documents

documents = load_documents(config.DOCUMENTS_DIR)
for name, text in documents:
    print(f"{name:40} {len(text)} znaków")

print()
print(documents[0][1])

01_praca_zdalna_v2.pdf                   831 znaków
02_podroze_sluzbowe_v3.pdf               784 znaków
03_podroze_sluzbowe_v1_ARCHIWUM.pdf      425 znaków
04_bezpieczenstwo_IT.pdf                 715 znaków
05_klasyfikacja_danych.pdf               792 znaków
06_urlopy_i_nieobecnosci.pdf             561 znaków
07_onboarding_IT.pdf                     729 znaków
08_zakupy_i_dostawcy.pdf                 687 znaków
09_incydenty_i_ciaglosc.pdf              697 znaków
10_reklamacje_klientow.pdf               705 znaków
11_polityka_AI.pdf                       849 znaków
12_benefity_FAQ.pdf                      677 znaków

Northstar Services - material rekrutacyjny (fikcyjny)
Strona 1
 Polityka pracy zdalnej
Status: AKTUALNY | Wersja: 2.1 | Obowiązuje od: 2026-01-15 | Właściciel: HR
Zasady ogólne
Pracownik może wykonywać pracę zdalną maksymalnie 3 dni w tygodniu, o ile charakter
stanowiska na to pozwala. Stałe dni pracy zdalnej ustalane są z przełożonym i mogą być zmieniane
z co najmniej jed

## 2. Chunking po sekcjach

Nagłówek sekcji to krótka linia bez kropki na końcu, która następuje po zakończonym zdaniu. Każda sekcja to jeden chunk z doklejonym „tytuł – sekcja”, żeby fragment był zrozumiały sam w sobie. Status z nagłówka (albo `ARCHIW` w nazwie pliku) ustawia flagę `archived`.

In [3]:
from src.chunker import chunk_document

for c in chunk_document(*documents[0]):
    print(c["id"], c["metadata"])
    print(c["text"])
    print()

01_praca_zdalna_v2.pdf::0 {'source': '01_praca_zdalna_v2.pdf', 'chunk_index': 0, 'section': 'Zasady ogólne', 'status': 'AKTUALNY', 'archived': False}
Polityka pracy zdalnej – Zasady ogólne
Pracownik może wykonywać pracę zdalną maksymalnie 3 dni w tygodniu, o ile charakter stanowiska na to pozwala. Stałe dni pracy zdalnej ustalane są z przełożonym i mogą być zmieniane z co najmniej jednodniowym wyprzedzeniem.

01_praca_zdalna_v2.pdf::1 {'source': '01_praca_zdalna_v2.pdf', 'chunk_index': 1, 'section': 'Praca zdalna z zagranicy', 'status': 'AKTUALNY', 'archived': False}
Polityka pracy zdalnej – Praca zdalna z zagranicy
Praca zdalna poza Polską wymaga wcześniejszej zgody HR oraz przełożonego. Standardowa zgoda może obejmować maksymalnie 10 dni roboczych w roku kalendarzowym. Wniosek należy złożyć co najmniej 10 dni roboczych przed planowanym wyjazdem.

01_praca_zdalna_v2.pdf::2 {'source': '01_praca_zdalna_v2.pdf', 'chunk_index': 2, 'section': 'Bezpieczeństwo', 'status': 'AKTUALNY', 'archiv

In [4]:
all_chunks = [c for name, text in documents for c in chunk_document(name, text)]
print("Chunków:", len(all_chunks))
print("Archiwalne:", sorted({c["metadata"]["source"] for c in all_chunks if c["metadata"]["archived"]}))
print("Długość chunków:", min(len(c["text"]) for c in all_chunks), "-", max(len(c["text"]) for c in all_chunks), "znaków")

Chunków: 37
Archiwalne: ['03_podroze_sluzbowe_v1_ARCHIWUM.pdf']
Długość chunków: 81 - 285 znaków


## 3. Embeddingi

bge-m3 zamienia tekst na wektor 1024 liczb o długości 1. Dla wektorów o długości 1 podobieństwo cosinusowe to zwykły iloczyn skalarny. Pytanie zadane innymi słowami („z domu”) powinno być najbliżej zdania o pracy zdalnej.

In [5]:
import numpy as np

from src.embedder import embed_texts

question = "Ile dni mogę pracować z domu?"
sentences = [
    "Pracownik może wykonywać pracę zdalną maksymalnie 3 dni w tygodniu.",
    "Standardowy limit noclegu w Polsce wynosi 650 PLN brutto za noc.",
    "MFA jest obowiązkowe dla poczty, VPN i systemów HR.",
]
q, *vectors = np.array(embed_texts([question, *sentences]))
print("Wymiar:", len(q), "| długość wektora:", round(float(np.linalg.norm(q)), 3))
for sentence, v in zip(sentences, vectors):
    print(f"{q @ v:.3f}  {sentence}")

Wymiar: 1024 | długość wektora: 1.0
0.668  Pracownik może wykonywać pracę zdalną maksymalnie 3 dni w tygodniu.
0.360  Standardowy limit noclegu w Polsce wynosi 650 PLN brutto za noc.
0.410  MFA jest obowiązkowe dla poczty, VPN i systemów HR.


## 4. Zapis w Chroma

Dokładnie to, co robi `python main.py reset` i `add`. Chunk w Chroma to ID `plik::numer`, tekst, embedding i metadane (po nich filtrujemy archiwum).

In [6]:
import argparse

from main import cmd_add, cmd_reset
from src.datastore import get_collection

cmd_reset(argparse.Namespace())
cmd_add(argparse.Namespace(path=config.DOCUMENTS_DIR))

collection = get_collection()
record = collection.get(ids=["02_podroze_sluzbowe_v3.pdf::1"], include=["documents", "metadatas"])
print()
print(record["metadatas"][0])
print(record["documents"][0])

Usunięto kolekcję.


  01_praca_zdalna_v2.pdf: 3 chunków
  02_podroze_sluzbowe_v3.pdf: 3 chunków


  03_podroze_sluzbowe_v1_ARCHIWUM.pdf: 3 chunków (archiwalny, pomijany przy wyszukiwaniu)
  04_bezpieczenstwo_IT.pdf: 3 chunków
  05_klasyfikacja_danych.pdf: 3 chunków


  06_urlopy_i_nieobecnosci.pdf: 3 chunków
  07_onboarding_IT.pdf: 3 chunków
  08_zakupy_i_dostawcy.pdf: 3 chunków


  09_incydenty_i_ciaglosc.pdf: 3 chunków
  10_reklamacje_klientow.pdf: 3 chunków


  11_polityka_AI.pdf: 4 chunków
  12_benefity_FAQ.pdf: 3 chunków
Zapisano 12 plików, 37 chunków.

{'archived': False, 'section': 'Hotele', 'chunk_index': 1, 'source': '02_podroze_sluzbowe_v3.pdf', 'status': 'AKTUALNY'}
Polityka podróży służbowych – Hotele
Standardowy limit noclegu w Polsce wynosi 650 PLN brutto za noc. W Warszawie, Krakowie, Gdańsku i Wrocławiu limit wynosi 800 PLN brutto za noc. Przekroczenie limitu wymaga akceptacji dyrektora jednostki.


## 5. Wyszukiwanie

`retrieve()` liczy embedding pytania i bierze z Chroma 5 najbliższych chunków, z filtrem `archived = False`. Score to 1 − odległość cosinusowa.

In [7]:
from src.retriever import retrieve

question = "Ile dni w tygodniu można standardowo pracować zdalnie?"
for h in retrieve(question):
    print(f"{h['score']:.3f}  {h['id']:35} {h['section']}")

0.754  01_praca_zdalna_v2.pdf::0           Zasady ogólne
0.667  01_praca_zdalna_v2.pdf::1           Praca zdalna z zagranicy
0.490  06_urlopy_i_nieobecnosci.pdf::0     Urlop wypoczynkowy
0.481  01_praca_zdalna_v2.pdf::2           Bezpieczeństwo
0.464  07_onboarding_IT.pdf::2             SLA


Dlaczego filtr archiwum jest potrzebny: bez niego archiwalna sekcja „Hotele” (500 PLN) jest prawie tak samo podobna do pytania jak aktualna. Embedding mierzy podobieństwo treści, a nie to, czy dokument obowiązuje.

In [8]:
question = "Jaki jest limit za nocleg w hotelu w Krakowie podczas delegacji?"
raw = collection.query(query_embeddings=embed_texts([question]), n_results=3)
print("Bez filtra:")
for meta, dist in zip(raw["metadatas"][0], raw["distances"][0]):
    print(f"  {1 - dist:.3f}  {meta['source']} ({meta['section']})")

print("Z filtrem (retrieve):")
for h in retrieve(question)[:3]:
    print(f"  {h['score']:.3f}  {h['source']} ({h['section']})")

Bez filtra:
  0.668  02_podroze_sluzbowe_v3.pdf (Hotele)
  0.592  03_podroze_sluzbowe_v1_ARCHIWUM.pdf (Hotele)
  0.400  02_podroze_sluzbowe_v3.pdf (Rezerwacje)
Z filtrem (retrieve):
  0.668  02_podroze_sluzbowe_v3.pdf (Hotele)
  0.400  02_podroze_sluzbowe_v3.pdf (Rezerwacje)
  0.373  06_urlopy_i_nieobecnosci.pdf (Urlop wypoczynkowy)


## 6. Prompt

Do modelu idzie prompt systemowy z zasadami (tylko kontekst, stałe zdanie odmowy, źródła na końcu) i wiadomość z fragmentami. Każdy fragment ma nagłówek `[źródło: plik, sekcja: …]`, z którego model przepisuje źródło.

In [9]:
from src.generator import build_messages

question = "Ile dni w tygodniu można standardowo pracować zdalnie?"
messages = build_messages(question, retrieve(question))
for m in messages:
    print(f"--- {m['role']} ---")
    print(m["content"])
    print()

--- system ---
Jesteś asystentem odpowiadającym na pytania pracowników na podstawie dokumentów firmowych.
Zasady:
- Odpowiadaj wyłącznie na podstawie podanego kontekstu. Niczego nie zmyślaj.
- Jeśli w kontekście nie ma odpowiedzi, odpowiedz dokładnie: "Nie wiem – w dostępnych dokumentach nie ma tej informacji." i nie podawaj źródeł.
- Odpowiadaj zwięźle, po polsku.
- Na końcu podaj źródła, z których faktycznie korzystałeś, w formacie: Źródła: plik (sekcja), plik (sekcja)

--- user ---
Kontekst:
[źródło: 01_praca_zdalna_v2.pdf, sekcja: Zasady ogólne]
Polityka pracy zdalnej – Zasady ogólne
Pracownik może wykonywać pracę zdalną maksymalnie 3 dni w tygodniu, o ile charakter stanowiska na to pozwala. Stałe dni pracy zdalnej ustalane są z przełożonym i mogą być zmieniane z co najmniej jednodniowym wyprzedzeniem.

[źródło: 01_praca_zdalna_v2.pdf, sekcja: Praca zdalna z zagranicy]
Polityka pracy zdalnej – Praca zdalna z zagranicy
Praca zdalna poza Polską wymaga wcześniejszej zgody HR oraz prze

## 7. Odpowiedź modelu

Surowa odpowiedź `qwen3:4b` (wariant Thinking) zaczyna się od rozumowania po angielsku zakończonego `</think>`. `generate_answer()` zostawia tylko tekst po tym znaczniku. Na CPU ta komórka trwa ok. 1 min, na GPU kilka sekund.

In [10]:
import ollama

response = ollama.chat(
    model=config.LLM_MODEL,
    messages=messages,
    think=False,
    options={"num_ctx": config.NUM_CTX, "temperature": config.TEMPERATURE, "seed": config.SEED,
             "num_predict": config.NUM_PREDICT},
)
raw = response.message.content
print(f"Tokeny promptu: {response.prompt_eval_count}, tokeny odpowiedzi: {response.eval_count}")
print("\n--- surowa odpowiedź (początek) ---")
print(raw[:500], "...")
print("\n--- po wycięciu rozumowania ---")
print(raw.rsplit("</think>", 1)[-1].strip())

Tokeny promptu: 715, tokeny odpowiedzi: 436

--- surowa odpowiedź (początek) ---
Okay, let's see. The user is asking how many days per week someone can standardly work remotely.

First, I need to check the provided context documents to find the answer. The context has several sources from different PDFs.

Looking at the first source: 01_praca_zdalna_v2.pdf, section "Zasady ogólne" (General Rules). It says: "Pracownik może wykonywać pracę zdalną maksymalnie 3 dni w tygodniu, o ile charakter stanowiska na to pozwala." So that translates to "Employee may perform remote work up  ...

--- po wycięciu rozumowania ---
Standardowo można pracować zdalnie maksymalnie 3 dni w tygodniu.

Źródła: 01_praca_zdalna_v2.pdf (Zasady ogólne)


Pytanie, na które w dokumentach nie ma odpowiedzi: model powinien odpowiedzieć stałym zdaniem odmowy, bez źródeł.

In [11]:
from src.generator import generate_answer

question = "Jaki jest numer telefonu do prezesa firmy?"
print(generate_answer(question, retrieve(question)))

Nie wiem – w dostępnych dokumentach nie ma tej informacji.


## 8. Ewaluacja

Pełna ewaluacja z LLM to `python -m eval.run_eval` (wyniki w `eval/results.md`). Tutaj tylko szybki test samego wyszukiwania: czy dla każdego pytania z odpowiedzią właściwy plik jest na 1. miejscu i czy wszystkie oczekiwane pliki są w top-5.

In [12]:
import json

questions = json.loads((ROOT / "eval" / "questions.json").read_text(encoding="utf-8"))
top1 = top5 = total = 0
for q in questions:
    if q["type"] == "none":
        continue
    sources = [h["source"] for h in retrieve(q["question"])]
    total += 1
    top1 += sources[0] in q["expected_sources"]
    top5 += set(q["expected_sources"]) <= set(sources)
print(f"Właściwy plik na 1. miejscu: {top1}/{total}, wszystkie oczekiwane pliki w top-5: {top5}/{total}")

Właściwy plik na 1. miejscu: 18/18, wszystkie oczekiwane pliki w top-5: 18/18
